# ROHub upload + SPARQL queries for the Hele-Shaw cells example

This notebook uploads the per-configuration `solution_field_data.zip`
files (which contain a hand-rolled `ro-crate-metadata.json` with
`m4i:`-namespaced predicates) to ROHub, then queries the SPARQL
endpoint to extract the `phase1_volume_fraction` metric for each
uploaded research object.

**Prerequisites**

- A ROHub account on the [dev](https://rohub2020-devel.apps.paas-dev.psnc.pl/)
  or [prod](https://www.rohub.org/) endpoint.
- The benchmark has been run: `python openfoam/run_benchmark.py` should
  have produced `results/<config>/solution_field_data.zip` for each
  configuration.
- The opt-in env: `mamba env create -n hs-rohub -f openfoam/environment_rohub.yml`.

**Note**: The SPARQL query in Cell 6 uses `m4i:` predicates
(`m4i:Method`, `m4i:hasParameter`, `m4i:investigates`,
`m4i:implementedByTool`) — the same shape the plate benchmark's
`rohub.md` documents. A single ROHub SPARQL endpoint can host both
benchmarks and the queries compose.

In [ ]:
import osimport jsonfrom pathlib import Pathfrom getpass import getpass# Toggle between dev (default) and prod. The endpoints differ in:#   - API_URL#   - KEYCLOAK_URL (and its client_id/secret)#   - SPARQL_ENDPOINTENDPOINT = "dev"  # or "prod"if ENDPOINT == "dev":    import rohub    rohub.settings.API_URL = "https://rohub2020-devel.apps.paas-dev.psnc.pl/api/"    rohub.settings.KEYCLOAK_CLIENT_ID = "rohub2020-cli"    rohub.settings.KEYCLOAK_CLIENT_SECRET = "714617a7-87bc-4a88-8682-5f9c2f60337d"    rohub.settings.KEYCLOAK_URL = "https://keycloak-dev.apps.paas-dev.psnc.pl/auth/realms/rohub/protocol/openid-connect/token"    rohub.settings.SPARQL_ENDPOINT = "https://virtuoso-rohub2020-devel.apps.bst2.paas.psnc.pl/sparql"elif ENDPOINT == "prod":    import rohub    rohub.settings.API_URL = "https://api.rohub.org/api/"    rohub.settings.KEYCLOAK_CLIENT_ID = "rohub2020-public-cli"    rohub.settings.KEYCLOAK_URL = "https://login.rohub.org/auth/realms/rohub/protocol/openid-connect/token"    rohub.settings.SPARQL_ENDPOINT = "https://virtuoso-rohub2020-production.apps.bst2.paas.psnc.pl/sparql"# Resolve where the results live.REPO_ROOT = Path("/Users/vasiliy/Documents/GitHub/V-V-Betty/hele-shaw-cells-example")  # edit if neededRESULTS_DIR = REPO_ROOT / "results"_globbed = sorted(REPO_ROOT.glob("parameters_*.json"))CONFIG_IDS = [p.name.split("_")[1].split(".")[0] for p in _globbed]# Fallback: hard-code the three configs we know aboutif not CONFIG_IDS:    CONFIG_IDS = ["1", "2", "3"]print(f"Endpoint: {ENDPOINT}")print(f"API URL:  {rohub.settings.API_URL}")print(f"SPARQL:   {rohub.settings.SPARQL_ENDPOINT}")print(f"Configs:  {CONFIG_IDS}")

In [ ]:
# Login. Read from env vars; fall back to interactive prompt.username = os.environ.get("ROHUB_USERNAME") or input("ROHub username: ")password = os.environ.get("ROHUB_PASSWORD") or getpass("ROHub password: ")rohub.login(username=username, password=password)print("Logged in.")

In [ ]:
# Find the solution_field_data.zip for each configuration.zips = {}for cfg in CONFIG_IDS:    candidate = RESULTS_DIR / cfg / "solution_field_data.zip"    if candidate.exists():        zips[cfg] = candidate    else:        print(f"WARNING: {candidate} not found; configuration {cfg} will be skipped")print(f"Found {len(zips)} zip(s) to upload: {list(zips.keys())}")

In [ ]:
uuids = {}for cfg, zip_path in zips.items():    print(f"Uploading {zip_path.name} ({cfg})...")    ro = rohub.ros_upload(path_to_zip=str(zip_path))    uuids[cfg] = ro.identifier    print(f"  -> UUID {ro.identifier}  (https://w3id.org/ro-id-dev/{ro.identifier})")# Persist for later cells(Path("notebooks") / "rohub_uuids.json").write_text(    json.dumps(uuids, indent=2))print(f"Wrote notebooks/rohub_uuids.json with {len(uuids)} UUIDs")

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper(rohub.settings.SPARQL_ENDPOINT)

rows = []
for cfg, uuid in uuids.items():
    # Construct the Dataset IRI from the UUID.
    ro_id_base = "ro-id-dev" if ENDPOINT == "dev" else "ro-id"
    dataset_iri = f"https://w3id.org/{ro_id_base}/{uuid}"

    # m4i-aware query. The ro_crate.py in this repo emits a m4i:Method
    # node with m4i:hasParameter (input), m4i:investigates (output)
    # and m4i:implementedByTool. We walk all three.
    query = f"""
    PREFIX schema: <http://schema.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX m4i: <https://w3id.org/nfdi4ing/metadata4ing#>
    SELECT DISTINCT ?param_label ?param_value
                    ?metric_label ?metric_value
                    ?tool_label
    WHERE {{
      GRAPH ?g {{ <{dataset_iri}> a schema:Dataset . }}
      GRAPH ?g {{
        ?method a m4i:Method ;
                m4i:hasParameter     ?param_pv ;
                m4i:investigates     ?metric_pv ;
                m4i:implementedByTool ?tool .
        ?param_pv a schema:PropertyValue ;
                  rdfs:label ?param_label ;
                  schema:value ?param_value .
        ?metric_pv a schema:PropertyValue ;
                   rdfs:label ?metric_label ;
                   schema:value ?metric_value .
        ?tool a schema:SoftwareApplication ;
              rdfs:label ?tool_label .
        FILTER (LCASE(STR(?tool_label)) = "openfoam-heleshawfoam")
      }}
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql_results = sparql.query().convert()
    if not sparql_results["results"]["bindings"]:
        print(f"WARNING: no m4i:Method bindings for {cfg} ({uuid})")
        continue

    # Pivot the long-format bindings into one row per config.
    row = {"configuration": cfg, "uuid": uuid}
    for b in sparql_results["results"]["bindings"]:
        row[b["param_label"]["value"]] = b["param_value"]["value"]
        mlabel = b["metric_label"]["value"]
        if mlabel.startswith("metric."):
            row[mlabel] = b["metric_value"]["value"]
    rows.append(row)

import pandas as pd
df = pd.DataFrame(rows)
df

In [ ]:
import matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as plt# Group by NPA so each mesh refinement gets its own lineif not df.empty:    for npa, sub in df.groupby("NPA"):        sub = sub.sort_values("flow_rate_m3_s")        plt.plot(sub["flow_rate_m3_s"], sub["phase1_volume_fraction"],                 marker="o", label=f"NPA = NPZ = {npa}")    plt.xlabel("Inlet volumetric flow rate (m^3/s)")    plt.ylabel("Phase-1 (air) volume fraction at endTime")    plt.title("Hele-Shaw: phase1 volume fraction vs. flow rate (from ROHub)")    plt.legend()    plt.grid(True, alpha=0.3)    plt.tight_layout()    plt.savefig("notebooks/rohub_phase1_vs_flowrate.png", dpi=100)    print("Saved notebooks/rohub_phase1_vs_flowrate.png")    plt.show()

## Troubleshooting- **Login fails with 401**: Check `ROHUB_USERNAME` and `ROHUB_PASSWORD`  are the **dev** endpoint's credentials (not production). The  notebook's default endpoint is the dev one.- **Upload fails with RO-Crate validation error**: Open the  `ro-crate-metadata.json` inside the offending  `solution_field_data.zip` and check that all `@id` values are valid  URIs (no leading spaces, no special characters).- **SPARQL returns no results**: The named graph may not be public yet.  Wait a few minutes after upload and re-run cell 6.- **Use this notebook on a fresh results tree**: Just re-run cells 4-7  in order. The UUIDs are persisted in `notebooks/rohub_uuids.json`  for reference.